In [1]:
# Install and import
!pip install torch -q

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import requests
import time

In [3]:
# Load dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text
chars = sorted(list(set(text)))
vocab_size = len(chars)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

def encode(s): return [stoi[c] for c in s]

data = torch.tensor(encode(text), dtype=torch.long)


In [4]:
# Hyperparameters
seq_len = 64
batch_size = 64
d_model = 256
n_heads = 4
n_layers = 4
ffn_dim = 4*d_model
lr = 3e-4
epochs = 3  # keep small for faster comparison

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [5]:
# Dataset

class CharDataset(Dataset):
    def __init__(self, data, seq_len):
        self.data = data
        self.seq_len = seq_len
    def __len__(self):
        return len(self.data) - seq_len
    def __getitem__(self, idx):
        x = self.data[idx:idx+seq_len]
        y = self.data[idx+1:idx+seq_len+1]
        return x, y

loader = DataLoader(CharDataset(data, seq_len), batch_size=batch_size, shuffle=True)

In [6]:
# RMSNorm

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-8):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        x = x / torch.sqrt(norm + self.eps)
        return self.scale * x

In [7]:
# Transformer block

class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, ffn_dim, norm_type="layernorm"):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)

        if norm_type == "rmsnorm":
            self.norm1 = RMSNorm(d_model)
            self.norm2 = RMSNorm(d_model)
        else:
            self.norm1 = nn.LayerNorm(d_model)
            self.norm2 = nn.LayerNorm(d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, ffn_dim),
            nn.GELU(),
            nn.Linear(ffn_dim, d_model)
        )

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        ffn_out = self.ffn(x)
        x = self.norm2(x + ffn_out)
        return x

In [8]:
# Model
class MiniGPT(nn.Module):
    def __init__(self, norm_type="layernorm"):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)

        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, ffn_dim, norm_type)
            for _ in range(n_layers)
        ])

        self.norm_f = RMSNorm(d_model) if norm_type=="rmsnorm" else nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        B, T = x.shape
        tok = self.token_emb(x)
        pos = self.pos_emb(torch.arange(T, device=x.device))
        x = tok + pos

        for block in self.blocks:
            x = block(x)

        x = self.norm_f(x)
        return self.head(x)

In [9]:
# Training
def train_model(norm_type):
    model = MiniGPT(norm_type).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    start_time = time.time()
    losses = []

    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()

    for epoch in range(epochs):
        total_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits.view(-1, vocab_size), yb.view(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(loader)
        losses.append(avg_loss)
        print(f"{norm_type} Epoch {epoch+1}: Loss = {avg_loss:.4f}")

    total_time = time.time() - start_time

    if device == "cuda":
        memory = torch.cuda.max_memory_allocated() / 1024**2  # MB
    else:
        memory = 0

    return losses, total_time, memory

In [ ]:
# Run both models
print("Training LayerNorm model...")
ln_losses, ln_time, ln_mem = train_model("layernorm")

print("\nTraining RMSNorm model...")
rms_losses, rms_time, rms_mem = train_model("rmsnorm")

Training LayerNorm model...


In [11]:
# Final comparisson
print("\n FINAL COMPARISON")
print("-"*50)
print(f"Training Time (sec): LayerNorm = {ln_time:.2f}, RMSNorm = {rms_time:.2f}")
print(f"Memory Usage (MB):   LayerNorm = {ln_mem:.2f}, RMSNorm = {rms_mem:.2f}")
print(f"Final Loss:          LayerNorm = {ln_losses[-1]:.4f}, RMSNorm = {rms_losses[-1]:.4f}")

# Stability (loss difference)
ln_diff = max(ln_losses) - min(ln_losses)
rms_diff = max(rms_losses) - min(rms_losses)

print(f"Loss Stability (lower is better):")
print(f"LayerNorm fluctuation = {ln_diff:.4f}")
print(f"RMSNorm fluctuation  = {rms_diff:.4f}")


 FINAL COMPARISON
--------------------------------------------------
Training Time (sec): LayerNorm = 419.18, RMSNorm = 476.48
Memory Usage (MB):   LayerNorm = 112.39, RMSNorm = 122.12
Final Loss:          LayerNorm = 0.0230, RMSNorm = 0.0232
Loss Stability (lower is better):
LayerNorm fluctuation = 0.0408
RMSNorm fluctuation  = 0.0372
